In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
df = pd.read_csv(r"/content/black_friday.csv").drop_duplicates()
df.sample()

,User_ID,Product_ID,Gender,Age,Occupation,City_Category,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase
7189,1001132,P00000642,M,18-25,12,A,1,1,1,6.0,16.0,15688.0


In [ ]:
df[["Product_Category_2","Product_Category_3"]]=df[["Product_Category_2","Product_Category_3"]].fillna(0)
df=df.dropna()
print(df.shape)
df.info()

(135528, 12)
<class 'pandas.core.frame.DataFrame'>
Index: 135528 entries, 0 to 135527
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   User_ID                     135528 non-null  int64  
 1   Product_ID                  135528 non-null  object 
 2   Gender                      135528 non-null  object 
 3   Age                         135528 non-null  object 
 4   Occupation                  135528 non-null  int64  
 5   City_Category               135528 non-null  object 
 6   Stay_In_Current_City_Years  135528 non-null  object 
 7   Marital_Status              135528 non-null  int64  
 8   Product_Category_1          135528 non-null  int64  
 9   Product_Category_2          135528 non-null  float64
 10  Product_Category_3          135528 non-null  float64
 11  Purchase                    135528 non-null  float64
dtypes: float64(3), int64(4), object(5)
memory usage: 13.4+ MB


In [ ]:
df[["Product_Category_1","Product_Category_2","Product_Category_3","Marital_Status","Occupation"]]=df[["Product_Category_1","Product_Category_2","Product_Category_3","Marital_Status","Occupation"]].astype("int")


In [ ]:
df["Age"].unique()

array(['0-17', '55+', '26-35', '46-50', '51-55', '36-45', '18-25'],
      dtype=object)

In [ ]:
cols = ["Age","Marital_Status","City_Category","Occupation"]
for i in cols:
  print("\n")
  print(df[i].value_counts())




Age
26-35    53574
36-45    27063
18-25    25094
46-50    11025
51-55     9577
55+       5350
0-17      3845
Name: count, dtype: int64


Marital_Status
0    80084
1    55444
Name: count, dtype: int64


City_Category
B    57079
C    41945
A    36504
Name: count, dtype: int64


Occupation
4     17907
0     17275
7     14497
1     11608
17     9749
20     8446
12     7514
14     6709
2      6375
16     6192
6      4838
3      4545
10     3317
11     3015
15     2945
5      2874
19     2211
13     1997
18     1575
9      1554
8       385
Name: count, dtype: int64


In [ ]:
df["Stay_In_Current_City_Years"]=df["Stay_In_Current_City_Years"].replace("4+","4")

In [ ]:
df["Stay_In_Current_City_Years"]=df["Stay_In_Current_City_Years"].astype("int")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 135528 entries, 0 to 135527
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   User_ID                     135528 non-null  int64  
 1   Product_ID                  135528 non-null  object 
 2   Gender                      135528 non-null  object 
 3   Age                         135528 non-null  object 
 4   Occupation                  135528 non-null  int64  
 5   City_Category               135528 non-null  object 
 6   Stay_In_Current_City_Years  135528 non-null  int64  
 7   Marital_Status              135528 non-null  int64  
 8   Product_Category_1          135528 non-null  int64  
 9   Product_Category_2          135528 non-null  int64  
 10  Product_Category_3          135528 non-null  int64  
 11  Purchase                    135528 non-null  float64
dtypes: float64(1), int64(7), object(4)
memory usage: 13.4+ MB


In [ ]:
df["User_ID"].value_counts().sum()

np.int64(135528)

In [ ]:
occupation_map = {    #it is done so in app user must put a proper occupation
    0: "Mechanic",
    1: "Doctor",
    2: "Teacher",
    3: "Artist",
    4: "Engineer",
    5: "Lawyer",
    6: "Chef",
    7: "Accountant",
    8: "Pilot",
    9: "Nurse",
    10: "Architect",
    11: "Scientist",
    12: "Electrician",
    13: "Plumber",
    14: "Journalist",
    15: "Farmer",
    16: "Salesperson",
    17: "Photographer",
    18: "Musician",
    19: "Police Officer",
    20: "Software Developer",
}

df["Occupation"] = df["Occupation"].map(occupation_map)

In [ ]:
city_map = {
    "A": "nagpur",
    "B": "pune",
    "C": "mumbai",}
df["City_Category"] = df["City_Category"].map(city_map)

In [ ]:
X=df.drop(columns=["Purchase","Product_ID","User_ID"])
y=df["Purchase"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn import preprocessing
preprocessing= ColumnTransformer([("scaling", StandardScaler(),["Product_Category_3","Product_Category_2","Product_Category_1","Stay_In_Current_City_Years","Marital_Status"]),
                        ("encoding", OneHotEncoder(),["City_Category","Gender","Age","Occupation"])])

trf=Pipeline([("preprocessing",preprocessing),
              ("model", RandomForestRegressor(n_estimators=250, max_depth=6))])

In [ ]:
trf.fit(X_train,y_train)
y_predict = trf.predict(X_test)
print("R²:", r2_score(y_test, y_predict))
print(root_mean_squared_error(y_test, trf.predict(X_test)))

R²: 0.623895822100649
3050.495748200247


In [ ]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(trf, f)